In [ ]:
# Celda 1: Entorno y carga
import runpy
from pathlib import Path
from pyspark.sql import functions as F

runpy.run_path(str(next(p / "notebooks" / "_init_env.py" for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "notebooks" / "_init_env.py").is_file())))

from src.catalog import EXTREMADURA_PROVINCE_CODES, GOLD_MITECO_ANNUAL_TABLE, SILVER_MITECO_TABLE

df_silver = spark.table(SILVER_MITECO_TABLE)
df_gold = spark.table(GOLD_MITECO_ANNUAL_TABLE)

In [ ]:
# Celda 2: Checks de calidad y coherencia Silver vs Gold
assert df_silver.count() > 0 and df_gold.count() > 0, "Tablas MITECO vacías"
assert df_silver.filter(~F.col("provincia").isin(list(EXTREMADURA_PROVINCE_CODES))).count() == 0, "Provincia inválida"
assert df_silver.count() == df_silver.dropDuplicates(["anio", "provincia", "num_parte"]).count(), "Partes duplicados"

total_silver = df_silver.count()
total_gold = df_gold.agg(F.sum("total_incendios")).collect()[0][0]
assert total_silver == total_gold, f"Descuadre Silver ({total_silver}) vs Gold ({total_gold})"
print("Checks MITECO OK. Total partes:", total_silver)

In [ ]:
# Celda 3: Resumen anual por provincia
display(df_gold.orderBy(F.desc("anio"), "provincia"))

In [ ]:
# Celda 4: Cobertura de municipio y coordenadas por año
display(
    df_silver.groupBy("anio")
    .agg(
        F.count("*").alias("partes"),
        F.sum(F.when(F.col("municipio").isNotNull(), 1).otherwise(0)).alias("con_municipio"),
        F.sum(F.when(F.col("latitud").isNotNull() & F.col("longitud").isNotNull(), 1).otherwise(0)).alias("con_coords"),
    )
    .orderBy("anio")
)